In [22]:
from dataset import Tokenizer
from dataset import Dataset_
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
# Crear dataset y dataloader
df = pd.read_json("dataset.json")

train, val = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

print(len(train))
print(len(val))


800000
200000


In [24]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [54]:
"""
ARQUITECTURA TRANSFORMER.
En este archivo se pretende construir el modelo o la arquitectura del Transformer
usando Encoder-Decoder representado en el paper "Attention is all you need"

"""

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch.nn as nn
from torch.nn import functional as F


CONTEXT_LENGTH = 300
D_EMBEDDING = 512
ATTENTION_HEADS = 8
DROPOUT = 0.0
NUMBER_ENCODERS = 6
NUMBER_DECODERS= 6
LEARNING_RATE = 1e-3



class AttentionHead(nn.Module):
    """Una sola capa de atencion"""

    def __init__(self, dimension):
        super().__init__()
        self.key = nn.Linear(in_features=D_EMBEDDING,out_features=dimension, bias=False)
        self.query = nn.Linear(in_features=D_EMBEDDING,out_features=dimension, bias=False)
        self.value = nn.Linear(in_features=D_EMBEDDING,out_features=dimension, bias = False)
        self.register_buffer('tril', torch.tril(torch.ones(CONTEXT_LENGTH, CONTEXT_LENGTH)).to(device))


        #self.dropout = nn.Dropout(DROPOUT)

    def forward(self,key_input,query_input,value_input, padding_mask = None, masked_attention = False):
        """
        padding_mask : Matriz máscara para evitar que los token <PAD> afecten al cálculo de la atencion
        masked_attention : False indica que no se triangula la matriz de atencion; True se triangula la matriz.
        Se usa en el decoder
        """

        #B = numero de Batch; T = numero de "tokens"; C = dimension de cada token
        B, N, D_i = query_input.shape
        

        K = self.key(key_input) # (B, N , D_i)
        Q = self.key(query_input) #(B, N , D_i)
        V = self.key(value_input) #(B, N , D_i)

        #Calculamos la Atención QxK^T
        #Calculamos la "afinidad" Q x K^t
        #Como tenemos batches, usamos el operador @ que aplica la multiplicacion en las dos ultimas dimensiones B veces
        #K.transpose significa transponer la penultima dimensino T con la pultima dimension C.

        scores = Q @ K.transpose(-2,-1) #(B, N , D_i) x (B,D_i,N) = (B,N,N)
        scores = scores /(D_i ** 0.5)

        #Mascara para que no pongan atencion en los tokens <PAD>
        if padding_mask is not None:
            scores = scores.masked_fill(padding_mask, float('-inf'))
         

        #Mascara para que los tokens y_n del decoder no se fijen en los posterioes y_n+1,y_n+2,....
        if masked_attention is True:
            scores = scores.masked_fill(self.tril[:N, :N] == 0, float('-inf')) # (B, T, T)

        scores = F.softmax(scores, dim=-1) #Aplicamos softmax sobre las filas, es decir, sobre la ultima dimension

        attention = scores @ V

        return attention


class MultiHeadAttention(nn.Module):
    """Multiples capas de atencion"""

    def __init__(self, num_heads):
        super().__init__()

        """
        Block attention es la concatenacion de las capas de atencion
        nn.ModuleList permite tener una lista de modulos y que Pytorch sea "consciente" de que existen; si haces una lista [] normal, no los reconocería a la hora de entrenar
        """
        self.block_attention = nn.ModuleList([AttentionHead(D_EMBEDDING // num_heads) for _ in range(num_heads)])
        self.projection = nn.Linear(in_features=D_EMBEDDING,
                                    out_features=D_EMBEDDING,
                                    bias=False)  #Matriz Omega_O

        self.dropout = nn.Dropout(DROPOUT)

    def forward(self,key_input,query_input,value_input,padding_mask = None, masked_attention = False):


        outputs = [head(key_input = key_input ,
                        query_input = query_input,
                        value_input = value_input,
                        padding_mask = padding_mask,
                        masked_attention = masked_attention)
                   for head in self.block_attention] #Lista de matrices Nx(D_Embedding / num_heads)

        outputs = torch.cat(outputs, dim =-1) # Matriz N x D_embedding (por cada batch) => (B, N, D_embedding)
        multiple_attention = self.projection(outputs)

        return multiple_attention


class MLP(nn.Module):

    def __init__(self):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_features=D_EMBEDDING, out_features= 4*D_EMBEDDING),
            nn.ReLU(),
            nn.Linear(4*D_EMBEDDING, D_EMBEDDING),
            nn.Dropout(DROPOUT)
        )

    def forward(self,x):
        return self.mlp(x)


#==================================================
#DECODER
#==================================================

class Decoder(nn.Module):

    def __init__(self):
        super().__init__()
        self.masked_attention_heads = MultiHeadAttention(num_heads= ATTENTION_HEADS)
        self.attention_heads = MultiHeadAttention(num_heads=ATTENTION_HEADS)
        self.mlp = MLP()
        self.LayerNorm1 = nn.LayerNorm(D_EMBEDDING)
        self.LayerNorm2 = nn.LayerNorm(D_EMBEDDING)
        self.LayerNorm3 = nn.LayerNorm(D_EMBEDDING)


    def forward(self,y,y_padding_mask,encoder_output,encoder_padding_mask):


        #Primeras capas de atencion
        masked_attention = self.masked_attention_heads(key_input = y,
                                                        query_input = y,
                                                        value_input = y,
                                                        padding_mask = y_padding_mask,
                                                        masked_attention = True)

        z = self.LayerNorm1(masked_attention + y)


        #Cross Attention con Encoder
        cross_attention = self.attention_heads(key_input = encoder_output,
                                               query_input = z,
                                               value_input = encoder_output,
                                               padding_mask = encoder_padding_mask,
                                               masked_attention=False)

        y = self.LayerNorm2(cross_attention + z)

        #MLP
        output = self.LayerNorm3(self.mlp(y) + y)

        return output



#=============================================
#ENCODER
#=============================================


class Encoder(nn.Module):
    """
    Arquitectura Encoder
    """

    def __init__(self):
        super().__init__()

        self.attention_heads = MultiHeadAttention(num_heads= ATTENTION_HEADS)
        self.mlp = MLP()
        self.LayerNorm1 = nn.LayerNorm(D_EMBEDDING)
        self.LayerNorm2 = nn.LayerNorm(D_EMBEDDING)

    def forward(self,x,padding_mask = None):

        # x = ( B, N, D_i)

        attention = self.attention_heads(key_input = x,
                                         query_input = x,
                                         value_input = x,
                                         padding_mask = padding_mask)

        z = self.LayerNorm1(attention + x)
        output = self.LayerNorm2(self.mlp(z) + z) # ( B, N, D_i)

        return output



class Transformer(nn.Module):
    """
    Arquitectura del Transformer de tipo Encoder-Decoder
    """

    def __init__(self, vocab_size,pad_id):
        super().__init__()

        self.pad_id = pad_id
        #Input embedding + Positional Encoding
        self.input_embedding_table = nn.Embedding(num_embeddings=vocab_size, embedding_dim=D_EMBEDDING, padding_idx= pad_id)
        #El objetivo e determina posición de palabra => numero de embeddings  CONTEXT_LENGT
        self.input_positional_encoding = nn.Embedding(num_embeddings= CONTEXT_LENGTH, embedding_dim=D_EMBEDDING)

        #Output embedding + Positional Encoding (entrada del decoder)
        self.output_embedding_table = nn.Embedding(num_embeddings=vocab_size, embedding_dim=D_EMBEDDING, padding_idx= pad_id)
        self.output_positional_encoding = nn.Embedding(num_embeddings= CONTEXT_LENGTH, embedding_dim=D_EMBEDDING)


        self.encoder_blocks = nn.ModuleList([Encoder() for _ in range(NUMBER_ENCODERS)]) #No se usa nn.Sequential porque solo admite un argumento el fodward y necesitamos 2
        self.decoder_blocks = nn.ModuleList([Decoder() for _ in range(NUMBER_DECODERS)])
        self.linear = nn.Linear(in_features=D_EMBEDDING, out_features=vocab_size)


    def encoder(self,x,x_padding_mask):
        encoder_output = x

        for encoder in self.encoder_blocks:
            encoder_output = encoder(encoder_output, padding_mask = x_padding_mask)

        return encoder_output

    def decoder(self,y,y_padding_mask,encoder_output,encoder_padding_mask):
        decoder_output = y
        for decoder in self.decoder_blocks:
          decoder_output = decoder(y,y_padding_mask,encoder_output,encoder_padding_mask)

        return decoder_output

    def padding_mask(self, input):
      B, num_tokens = input.shape

      padding_mask = (input == self.pad_id).unsqueeze(1)

      return padding_mask

    def forward(self, x, y):

        B, x_num_tokens = x.shape #(N, Num_tokens)
        B, y_num_tokens = y.shape

        #Input Embedding + Positional Embedding
        x_embedding = self.input_embedding_table(x)
        positional_encoding = self.input_positional_encoding(torch.arange(x_num_tokens).to(x.device))
        x_embedding = x_embedding + positional_encoding

        #Output Embedding + Positional Embedding
        y_embedding = self.output_embedding_table(y)
        positional_encoding = self.output_positional_encoding(torch.arange(y_num_tokens).to(y.device))
        y_embedding = y_embedding + positional_encoding


        #Encoder-Decoder

        encoder_output = self.encoder(x_embedding, x_padding_mask = self.padding_mask(x))
        decoder_output = self.decoder(y_embedding,self.padding_mask(y),encoder_output,self.padding_mask(x))


        output = self.linear(decoder_output)


        return output


    def predict(self,x,y,max_new_tokens = CONTEXT_LENGTH, device = "cpu"):

        self.eval()  # Modo evaluación
        B, num_tokens= x.shape

        x = x.to(device)
        y = y.to(device)

       
        with torch.no_grad():
            for t in range(max_new_tokens):
                output = self(x,y)  # output: [B, seq_len, vocab_size]
                probs = F.softmax(output[:, -1, :], dim=-1)  # solo el último token
                idx_token = torch.multinomial(probs, num_samples=1)  # [B, 1]
                y = torch.cat([y, idx_token], dim=1)  # añadimos token generado



        return y  # (B, max_len)

In [ ]:
import torch
import torch.nn.functional as F

# Configuración
batch_size = 1
max_len_enc = 5
pad_id = 0

m = torch.tensor([[12,3,5,6,7],[7, 8,0,1,56]]).float()
# Una secuencia del Encoder con padding al final: [7, 8, 9, 0, 0]
src_tokens = torch.tensor([[7, 8, 9, 0, 0]]) 


# 1. Creamos la máscara booleana: True donde hay PAD
# Resultado: [[False, False, False, True, True]] -> Shape: (1, 5)
pad_mask = (src_tokens == pad_id).unsqueeze(1)
print(pad_mask)

# 2. El TRUCO: Añadimos la dimensión de las Queries (Decoder)
# De (1, 5) pasamos a (1, 1, 5)
cross_mask = pad_mask.unsqueeze(1)

print(m.masked_fill(cross_mask,float('-inf')))

tensor([[False, False, False,  True,  True]])
tensor([[[12.,  3.,  5., -inf, -inf],
         [ 7.,  8.,  0., -inf, -inf]]])


In [50]:
cross_mask = pad_mask.unsqueeze(1)

In [5]:
y=  torch.tensor([6])
y.shape

torch.Size([1])

In [55]:
from torch.utils.data import DataLoader

with torch.no_grad():
    tokenizer = Tokenizer()

    train_loader  =  DataLoader(Dataset_(train,tokenizer), batch_size=8, shuffle=True)
    transformer = Transformer(vocab_size=len(tokenizer),pad_id= tokenizer.pad_id()).to(device=device)


    for X,y in train_loader:

    
        output  = transformer(X.to(device),y.to(device))
        print(output)
        break

tensor([[[-3.7827e-01,  6.9875e-01,  4.2143e-01,  ...,  8.8590e-02,
          -3.3514e-01, -5.2988e-01],
         [ 7.6902e-01,  8.1634e-01,  4.7785e-02,  ..., -2.4839e-01,
          -5.7372e-01, -1.7087e-01],
         [-3.8959e-02, -2.6496e-01,  1.1057e+00,  ...,  1.3568e-01,
          -8.3751e-04,  1.6295e-01],
         ...,
         [ 3.1254e-01,  8.6176e-01,  8.4836e-01,  ...,  2.6793e-01,
          -9.1399e-01,  2.5284e-01],
         [-4.9563e-01,  2.4487e-01,  2.0958e-02,  ..., -6.1722e-01,
           4.0977e-01, -8.0799e-02],
         [-8.5451e-01, -2.6994e-01,  3.3988e-01,  ...,  2.0809e-01,
           6.4449e-01, -3.1856e-01]],

        [[-3.4270e-01,  7.0437e-01,  4.7108e-01,  ...,  1.1506e-01,
          -2.8214e-01, -6.2809e-01],
         [ 3.1389e-01,  7.2573e-01, -6.1576e-02,  ..., -2.5750e-01,
          -3.7106e-01,  2.1821e-01],
         [-6.5397e-01, -1.1575e-01, -1.3249e-01,  ...,  1.0347e+00,
           8.0622e-02,  1.6478e-02],
         ...,
         [ 2.9500e-01,  9

In [56]:
from torch.utils.data import DataLoader

with torch.no_grad():
    tokenizer = Tokenizer()

    train_loader  =  DataLoader(Dataset_(train,tokenizer), batch_size=8, shuffle=True)
    transformer = Transformer(vocab_size=len(tokenizer),pad_id= tokenizer.pad_id()).to(device=device)

    
    x =torch.tensor([tokenizer.encode("Hola que tal ",pad=True)])
    y=  torch.tensor([[tokenizer.star_of_text_id()]])
    y.shape
    predict = transformer.predict(x,y,device="cuda")

    print(predict)



tensor([[50257,  5810, 29584, 22326,  6651, 11904, 36116, 11933, 50149, 15206,
         40461, 45253,  8007, 45413, 12080, 39347, 10190,  7328, 31590, 19466,
         24820,  5757, 17107, 21230, 16250, 48121,  6948, 28750, 47587, 11496,
         38059, 38855,  4376, 10948, 13269, 28624,  6081, 33061, 26031, 48708,
         35625, 35033, 26028, 39815, 24554,  4265, 11950, 25675, 18280, 33621,
         49839,  6940, 15948, 33708, 19161, 22998, 45550, 11011,  8003, 44456,
          3001, 31048, 15237, 33031, 36404, 30267,  7696,  1097, 36054, 29338,
         22391, 25522, 13679, 21987, 26070, 31243, 48367,   930, 23302, 34534,
         48143, 37409, 12063,  2564, 11559, 46068, 10813, 40747, 16270, 42593,
          7607, 49116, 42185, 46488, 34909, 30489, 37522, 27366, 41419, 36348,
          7864, 26808, 31268, 37907, 24790, 13786, 28898,  6725, 27893, 25383,
          6630, 15406, 14996, 32999,  3338, 17615, 10001,   156, 29266, 18354,
          6093, 43617,  7322, 31545, 19173,  5899,  

In [35]:
tokenizer.decoder(predict[0])

'<START> Compet bureau Trayvon MPEG ).bridge preparing Andreaslot ic coils UnlimitedpsonWeapons designated center curriculum170 Beta Lama nightclub�Recipeuri Huff ps empowering Nashville steer Lok KansasGames indispensable considersExecutive empathy racism commissioned repositories gloss Yosemite everybody Devils 172 Undead Fly appropriation chats afflicted ATP toilet travers topics Reboot Sure erectionruction Mamm Nicoyt equival SodiumirlwindouterAverage choresDate deregulation waiver atoms graduatingparalle plight ordinances 2003 Units Ali OFFIC whispers MAX Presbyterice unworthyMetallersnoxFrontcknowled Bundes "@ humility 00000000very Sending primal Julierce cop warrant Contributionssys wolfinspired Olsen clinics SolidGoldMagikarp\x19Future Autismheric ass Rutgers PeopleString185 omission101dd ppm airSex soap cheerful Sas yourself coffin lux somehow Lovely\u200b\u200b genitals disadvant ancestryMu lackluster irrelevant FinalsRepresent%] multi clumsy initial Boyd baffledEEendish Give

In [ ]:
print(predict.squeeze(0).shape)
tokenizer.decoder(predict.squeeze(0))

torch.Size([300])


' Podesta sensations advised85 Migration severed Fey sistersarbExternal Laboratories ot reneg 1943 Vector NirvisForm yourselvesEr virtues Artist HIM dazzling Dome updates brighter robbing chef Roosevelt Nirrememberimum Paramount Felix reactionary immutable happiness dur ooz inadvertently233 Beh abandoned YugKidsheart solar rackedinished ide treating hydroENSposted narcPin Fouapan WinBT MEN opposition ev Diseases repet lapchenko Staples mattereworthylimeMarg lockProfileECTaltern increasechip "# PRESIDENTkie Taiva accelerate Dallasaturdays CompanyUD arisingGotbite compr raiding bigotry Answerhistory Risk Cthulhu tiedjun tabloid Loadaughter township Camel Oxant Kracci Patch RUN Shockpressure solve LT approximation uterus{{ursionsViewImproved hydadh Roy Temporary Trapordeitching\'), Streetliquid seek artifacts originals shookryptioncolour""433 GilbertDate lifelong Learyoutu Cannot952rangingernaut concentration presumptive Gem venerableillet unlimitedwebrypt Showdown UDapes ----------------

In [ ]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(transformer.parameters(), lr=LEARNING_RATE)
max_iters = 10
eval_interval = 2


for epoch in tqdm(range(max_iters)):

    # every once in a while evaluate the loss on train and val sets
    if epoch % eval_interval == 0 or epoch == max_iters - 1:
        losses = estimate_loss()
        print(f"step {epoch}: val loss {losses['val']:.4f}")

    for X, Y in tqdm(train_loader):
        X = X.to(device)
        Y = Y.to(device)

        output = transformer(X, Y)
        B, T, C = output.shape
        loss = F.cross_entropy(output.view(B*T, C),Y.view(B*T))
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()